# 08 Feature Lifestyle Only

In [ ]:
# =========================
# FULL WORKING CELL (ALL-IN-ONE)
# - Fix MemoryError for matplotlib savefig
# - Defines load_preprocess_dataset/split_data/log_common_params
# - Runs MLflow nested runs for lifestyle-only experiments
# =========================

import os
os.environ["PYTHONWARNINGS"] = "ignore"
import warnings
warnings.simplefilter("ignore")

import gc
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")  # IMPORTANT: before pyplot
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, label_binarize
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_auc_score, roc_curve
)
from mlflow.models import infer_signature

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
except ModuleNotFoundError:
    SMOTE = None
    ImbPipeline = None


# -------------------------
# CONSTANTS / PATHS
# -------------------------
RANDOM_STATE = 26
TEST_SIZE = 0.2
EXPERIMENT_NAME = "Current Stress - Experiments"

def resolve_repo_root():
    cwd = Path.cwd().resolve()
    for parent in [cwd, *cwd.parents]:
        if (parent / "nostressia-machine-learning").exists() and (parent / "nostressia-backend").exists():
            return parent
    return cwd

REPO_ROOT = resolve_repo_root()
TRACKING_URI = "file:" + str((REPO_ROOT / "mlruns").resolve()).replace("\\", "/")
DATASET_PATH = REPO_ROOT / "nostressia-machine-learning" / "Current-Stress" / "datasets" / "raw" / "student_lifestyle_dataset.csv"

# If you run from notebook folder, set this accordingly:
NOTEBOOK_PATH = Path.cwd() / "08_feature_lifestyle_only.ipynb"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)


# -------------------------
# DATA PREP
# -------------------------
def categorize_academic_performance(gpa):
    if gpa >= 3.5:
        return "Excellent"
    elif 3.0 <= gpa < 3.5:
        return "Good"
    elif 2.0 <= gpa < 3.0:
        return "Fair"
    return "Poor"

def load_preprocess_dataset(dataset_path=DATASET_PATH):
    df = pd.read_csv(dataset_path)
    raw_df = df.copy()

    df["Academic_Performance"] = df["GPA"].apply(categorize_academic_performance)
    mapping_stress = {"Low": 0, "Moderate": 1, "High": 2}
    mapping_performance = {"Poor": 0, "Fair": 1, "Good": 2, "Excellent": 3}

    df["Stress_Level_Encoded"] = df["Stress_Level"].map(mapping_stress)
    df["Academic_Performance_Encoded"] = df["Academic_Performance"].map(mapping_performance)
    df = df.drop(columns=["Stress_Level", "Academic_Performance"])

    feature_cols = [c for c in df.columns if c not in ["Stress_Level_Encoded", "Student_ID"]]
    X = df[feature_cols].copy()
    y = df["Stress_Level_Encoded"].copy()
    return raw_df, df, X, y

def split_data(X, y):
    return train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

def log_common_params(feature_set, imbalance_handling="no", tuning_strategy="none"):
    mlflow.log_params({
        "dataset_path": str(DATASET_PATH),
        "feature_set": feature_set,
        "preprocessing": "Academic_Performance derivation + ordinal encoding + Student_ID dropped",
        "scaler": "RobustScaler for LR/stacking where applicable",
        "imputation": "none",
        "split_test_size": TEST_SIZE,
        "split_random_state": RANDOM_STATE,
        "split_stratify": True,
        "imbalance_handling": imbalance_handling,
        "tuning_strategy": tuning_strategy,
        "tracking_uri": TRACKING_URI,
    })


# -------------------------
# EVAL (RAM-FRIENDLY)
# -------------------------
def compute_metrics(y_true, y_pred, y_proba=None):
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "balanced_accuracy": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
    }
    if y_proba is not None:
        try:
            metrics["roc_auc"] = float(roc_auc_score(y_true, y_proba, multi_class="ovr"))
        except Exception:
            pass
    return metrics

def _safe_savefig(fig, out_path: Path, dpi: int = 80):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    plt.close("all")
    gc.collect()

def build_eval_artifacts(model, X_test, y_test, artifact_dir, prefix="eval"):
    artifact_dir = Path(artifact_dir)
    artifact_dir.mkdir(parents=True, exist_ok=True)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test) if hasattr(model, "predict_proba") else None

    metrics = compute_metrics(y_test, y_pred, y_proba)

    # Confusion matrix (lightweight)
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay(confusion_matrix=cm).plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(f"{prefix} - Confusion Matrix")
    fig.tight_layout()
    _safe_savefig(fig, artifact_dir / f"{prefix}_confusion_matrix.png", dpi=80)

    # ROC curve (micro-average) if proba exists
    if y_proba is not None:
        classes = np.unique(np.asarray(y_test))
        y_bin = label_binarize(y_test, classes=classes)
        try:
            if y_bin.shape[1] == 1 and y_proba.ndim == 2 and y_proba.shape[1] == 2:
                fpr, tpr, _ = roc_curve(y_test, y_proba[:, 1])
            else:
                fpr, tpr, _ = roc_curve(y_bin.ravel(), y_proba.ravel())

            fig, ax = plt.subplots(figsize=(5, 4))
            ax.plot(fpr, tpr, label="ROC micro-average")
            ax.plot([0, 1], [0, 1], linestyle="--")
            ax.set_xlabel("FPR")
            ax.set_ylabel("TPR")
            ax.set_title(f"{prefix} - ROC Curve")
            ax.legend(loc="lower right")
            fig.tight_layout()
            _safe_savefig(fig, artifact_dir / f"{prefix}_roc_curve.png", dpi=80)
        except Exception:
            pass

    (artifact_dir / f"{prefix}_classification_report.txt").write_text(
        classification_report(y_test, y_pred, digits=4, zero_division=0),
        encoding="utf-8"
    )

    pd.DataFrame({"y_true": np.asarray(y_test), "y_pred": np.asarray(y_pred)}).head(100).to_csv(
        artifact_dir / f"{prefix}_sample_predictions.csv", index=False
    )

    plt.close("all")
    gc.collect()
    return metrics


# -------------------------
# TRAINING LOOP
# -------------------------
raw_df, processed_df, X_all, y = load_preprocess_dataset()

feature_group = "lifestyle"
features = ['Sleep_Hours_Per_Day', 'Social_Hours_Per_Day', 'Physical_Activity_Hours_Per_Day']

X = X_all[features].copy()
X_train, X_test, y_train, y_test = split_data(X, y)

runs = [
    ('LifestyleOnly - LR',
     Pipeline(steps=[
         ('scaler', RobustScaler()),
         ('lr', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
     ]),
     'none', 'none'),

    ('LifestyleOnly - DT',
     DecisionTreeClassifier(random_state=RANDOM_STATE),
     'none', 'none'),

    ('LifestyleOnly - RF Default',
     RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
     'none', 'none'),
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

tuned_rf = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid={
        'n_estimators': [150, 300, 450],
        'max_depth': [None, 6, 10],
        'min_samples_split': [2, 4],
        'class_weight': [None, 'balanced']
    },
    scoring='f1_weighted',
    cv=cv,
    n_jobs=-1
)

smote_rf = ImbPipeline(steps=[
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('rf', RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        n_estimators=300,
        class_weight='balanced'
    ))
])

all_models = runs + [
    ('LifestyleOnly - RF Tuned', tuned_rf, 'none', 'grid_search'),
    ('LifestyleOnly - RF + SMOTE', smote_rf, 'smote', 'none')
]

with mlflow.start_run(run_name='LifestyleOnly - Experiments'):
    mlflow.set_tags({'module': 'current-stress', 'feature_group': feature_group})
    mlflow.log_param('subset_features', json.dumps(features))

    for run_name, estimator, imbalance, tune in all_models:
        with mlflow.start_run(run_name=run_name, nested=True):
            mlflow.set_tags({'module': 'current-stress', 'feature_group': feature_group})
            log_common_params(feature_set=feature_group,
                              imbalance_handling=imbalance,
                              tuning_strategy=tune)

            mlflow.log_param('model_name', run_name)
            mlflow.log_param('features', json.dumps(features))

            estimator.fit(X_train, y_train)
            model = estimator.best_estimator_ if hasattr(estimator, 'best_estimator_') else estimator

            if hasattr(estimator, 'best_params_'):
                mlflow.log_params({f'best_{k}': v for k, v in estimator.best_params_.items()})
                mlflow.log_metric('best_cv_score', float(estimator.best_score_))

            with tempfile.TemporaryDirectory() as td:
                art = Path(td)
                metrics = build_eval_artifacts(model, X_test, y_test, art, prefix='test')
                mlflow.log_metrics(metrics)

                sig = infer_signature(X_train.head(20), model.predict(X_train.head(20)))
                mlflow.sklearn.log_model(model, 'model', signature=sig, input_example=X_train.head(5))

                # Log notebook only if it exists (avoid crash)
                if NOTEBOOK_PATH.exists():
                    mlflow.log_artifact(str(NOTEBOOK_PATH), artifact_path='code')

                mlflow.log_artifacts(str(art), artifact_path='artifacts')

            gc.collect()

print("DONE ✅ Check MLflow UI (mlflow ui) and open http://127.0.0.1:5000")

2026/02/20 13:20:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/20 13:20:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/20 13:20:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/20 13:21:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/20 13:22:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


DONE ✅ Check MLflow UI (mlflow ui) and open http://127.0.0.1:5000
